 # Crazyflie ToF Measurements
 This notebook aims to analyze the ToF measurements done by the Crazyflie

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
from scipy.spatial.transform import Rotation as R

actual_tilt_deg = np.linspace(0, 60, 500)
altitudes = np.full_like(actual_tilt_deg, 1.0)
quats_xyzw = R.from_euler('y', actual_tilt_deg.reshape(-1, 1), degrees=True).as_quat()
quaternions = np.roll(quats_xyzw, 1, axis=1)


# ==========================================
# Code from synthetic sensor
# ==========================================
rotations = R.from_quat(quaternions, scalar_first=True)
r_zz = rotations.as_matrix()[:, 2, 2]  # Z-Z component of the rotation
r_zz_safe = np.clip(r_zz, -1.0, 1.0)  # fix for floating point errors
drone_tilt_angle = np.arccos(r_zz_safe) # actual angle of the drone
tilt_raw = np.abs(drone_tilt_angle) - math.radians(7.5) # tilt compensated for cone.
tilt = np.maximum(0.0, tilt_raw)  # altitudes grow if drone is tilted more than 7.5 degrees. 0 if drone is tilted less than 7.5 degrees.
distances = altitudes / np.cos(tilt)
# ==========================================

true_tilt_rad = np.radians(actual_tilt_deg)
true_distances = altitudes / np.cos(true_tilt_rad)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(actual_tilt_deg, distances, label='Firmware Logic (with 7.5° offset)', color='#1f77b4', linewidth=3)

plt.plot(actual_tilt_deg, true_distances, label='True Geometrical Distance', color='#ff7f0e', linestyle='--', linewidth=2)

plt.axvline(7.5, color='gray', linestyle=':', label='7.5° Deadband Threshold')

plt.title('Sensor Distance vs. Drone Tilt Angle', fontsize=14, pad=15)
plt.xlabel('Actual Drone Tilt Angle (degrees)', fontsize=12)
plt.ylabel('Calculated Sensor Distance (meters)', fontsize=12)
plt.xlim(0, 20)
plt.ylim(0.95, 1.1)
plt.legend(fontsize=10, loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()